<a href="https://colab.research.google.com/github/fjolivos/WAPOR_workshop/blob/main/WAPOR_Workshop_Mexico_City.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# D-Efficient Factorial Vignette Design

This notebook provides a step-by-step guide to selecting vignettes from the complete vignette universe using a D-efficient experimental design. D-efficiency is a widely used measure of design quality that considers balance and orthogonality simultaneously. A more D-efficient design provides more precise estimates of the effects of the vignette dimensions while limiting correlations among them.

The complete vignette universe contains every possible combination of the levels of eight dimensions: gender, age, education, job tenure, number of children, job performance, the firm’s economic situation, and monthly gross earnings. Because it is not feasible to present all possible combinations to respondents, we will use a D-efficient algorithm to select a smaller and statistically informative subset.

The final design contains 160 vignettes, organized into 10 decks of 16 vignettes each. The code first generates multiple candidate designs and compares their D-efficiency and correlations. It then selects a near-optimal design with low correlations among the vignette dimensions. For simplicity, the 160 selected vignettes are subsequently allocated randomly across the 10 decks.

Finally, the notebook transforms the selected combinations into natural-language vignette descriptions, creates numeric variables identifying the level of each dimension, and exports the final design to an Excel file.

For a classic introduction to experimental designs in factorial surveys, see:

Dülmer, H. (2007). Experimental plans in factorial surveys: random or quota design?. *Sociological Methods & Research*, 35(3), 382-409.


**Setting up the R environment**

We begin by installing and loading the R packages required for the exercise. The packages only need to be installed once, but they must be loaded each time a new R session begins.

AlgDesign generates the D-efficient experimental design.
dplyr supports data manipulation and organization.
purrr facilitates repeated operations across candidate designs.
openxlsx exports the final vignettes to an Excel file.

Finally, set.seed() ensures that the random components of the procedure are reproducible. Running the notebook with the same seed and settings will produce the same results.

In [ ]:
# Run once if the packages are not installed:
install.packages(c("AlgDesign", "dplyr", "purrr", "openxlsx", "tidyr"))

# Load the packages

library(AlgDesign)
library(dplyr)
library(purrr)
library(openxlsx)
library(tidyr)

set.seed(20260825)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




**Defining the vignette universe**

We first define the dimensions of the factorial survey and the levels that each dimension can take. In this example, every vignette describes a person according to eight dimensions: gender, age, education, job tenure, number of children, job performance, the firm’s economic situation, and monthly gross earnings.

The order of the levels is important because it will later determine their numeric coding. For example, age 30 will be coded as 1, age 40 as 2, age 50 as 3, and age 60 as 4.

The expand.grid() function then combines every level of every dimension to create the complete vignette universe. With the dimensions specified here, the total number of possible combinations is:

$$ 2 \times 4 \times 2 \times 2 \times 4 \times 2 \times 3 \times 5 = 3{,}840 $$

Therefore, the full universe contains 3,840 unique vignettes. A unique identification number is added to each combination so that every vignette can be tracked throughout the design-selection process.

In [ ]:

# ============================================================
# 1. DEFINE THE FACTORS AND LEVELS
# ============================================================

# Modify the earnings values if you use different amounts.
earnings_levels <- c(
  500, 1500, 3000, 6000, 15000
)

factors <- list(
  gender = factor(
    c("Male", "Female"),
    levels = c("Male", "Female")
  ),

  age = factor(
    c("30", "40", "50", "60"),
    levels = c("30", "40", "50", "60")
  ),

  education = factor(
    c("Vocational training", "University degree"),
    levels = c("Vocational training", "University degree")
  ),

  tenure = factor(
    c("Short", "Long"),
    levels = c("Short", "Long")
  ),

  children = factor(
    c("0", "1", "2", "3"),
    levels = c("0", "1", "2", "3")
  ),

  performance = factor(
    c("Below average", "Above average"),
    levels = c("Below average", "Above average")
  ),

  firm_situation = factor(
    c("High profits", "Solid", "Threatened by bankruptcy"),
    levels = c(
      "High profits",
      "Solid",
      "Threatened by bankruptcy"
    )
  ),

  earnings = factor(
    earnings_levels,
    levels = earnings_levels
  )
)

# Create the full factorial universe:
# 2 × 4 × 2 × 2 × 4 × 2 × 3 × 5 = 3,840
vignette_universe <- expand.grid(
  factors,
  KEEP.OUT.ATTRS = FALSE,
  stringsAsFactors = TRUE
)

vignette_universe$vignette_universe_id <-
  seq_len(nrow(vignette_universe))

nrow(vignette_universe)
# Expected result: 3,840



[1] 3840

**Specifying the model to be optimized**

A D-efficient design is optimized for a particular statistical model. We must therefore specify in advance which effects the final design should allow us to estimate efficiently.

The model below contains only the main effects of the eight vignette dimensions. It is appropriate when the primary objective is to estimate the independent effect of each dimension on respondents’ evaluations. For example, it allows us to examine whether earnings evaluations differ according to gender, age, education, performance, or earnings while holding the other vignette characteristics constant.
If the research hypotheses involve interactions, these should be included in the formula before generating the design. An interaction examines whether the effect of one dimension depends on another dimension-. You may create the variable or add the interaction (:).

In this design, earnings is treated as a categorical factor. The model therefore estimates separate effects for the five earnings levels without assuming that the relationship between earnings and the outcome is linear. In particular, it does not assume that the difference between €500 and €1,500 has the same effect as the difference between €6,000 and €15,000.

In [ ]:
# ============================================================
# 2. SPECIFY THE MODEL TO BE OPTIMIZED
# ============================================================

# Main-effects model.

design_formula <- ~
  gender +
  age +
  education +
  tenure +
  children +
  performance +
  firm_situation +
  earnings

**Evaluating candidate designs**

Not every subset of 160 vignettes provides an equally informative design. Some may overrepresent particular factor levels or create dependencies between dimensions, making their effects harder to estimate separately.

We compare candidate designs using D-efficiency, which considers balance and orthogonality jointly. Higher values indicate that the model parameters can be estimated more precisely.

The function first converts the categorical dimensions into a numerical design matrix. It then calculates D-efficiency from the determinant of the information matrix, \(X'X\). If the determinant is non-positive, the function returns zero, indicating that not all model parameters can be estimated independently.

In [ ]:

# ============================================================
# 3. FUNCTIONS TO EVALUATE CANDIDATE DESIGNS
# ============================================================

calculate_d_efficiency <- function(data, formula) {

  # Create the numerical design matrix.
  x <- model.matrix(
    formula,
    data = data,
    contrasts.arg = lapply(
      data[names(factors)],
      function(x) contr.sum(nlevels(x))
    )
  )

  # Calculate the information matrix, X'X.
  information_matrix <- crossprod(x)

  # Store the number of model parameters and vignettes.
  p <- ncol(x)
  n <- nrow(x)

  # Calculate the determinant using its logarithmic form
  # to avoid numerical problems with very large values.
  determinant_value <- determinant(
    information_matrix,
    logarithm = TRUE
  )

  # A non-positive determinant indicates that the model
  # parameters cannot all be estimated independently.
  if (determinant_value$sign <= 0) {
    return(0)
  }

  # Calculate the scaled D-efficiency.
  exp(
    determinant_value$modulus / p
  ) / n
}


**Searching for an efficient 160-vignette design**

It is not computationally feasible to evaluate every possible selection of 160 vignettes from a universe of 3,840—there are approximately \(10^{276}\) possible designs. The Fedorov algorithm instead searches for a highly D-efficient design by repeatedly exchanging selected and unselected vignettes when an exchange improves D-efficiency.

**Repeating the search**

The optimization is repeated 200 times. Each repetition produces one complete candidate design containing 160 vignettes. More searches increase the opportunity to find a better design but also increase computation time. For a demonstration, 20–50 searches may be sufficient; for a final design, 200–500 are reasonable.

**Optimization settings**

The main arguments passed to optFederov() are:

frml: specifies the model for which the design is optimized.

*   frml: specifies the model for which the design is optimized.
*   data: supplies the complete vignette universe.
*   nTrials = 160: selects 160 vignettes for each candidate design.
*   approximate = FALSE: requests an exact design containing actual vignette    combinations.
*   criterion = "D": optimizes the design according to D-efficiency.
*   maxIteration = 200: allows up to 200 exchange iterations within each search.
*   nRepeats = 1: performs one optimization attempt in each loop repetition because the outer loop already repeats the procedure 200 times.

Each candidate set of 160 vignettes is stored in candidate_designs.

The statistics are stored in design_diagnostics, with one row for each candidate design. This allows the 200 alternatives to be compared after the search is complete.

In [ ]:
# ============================================================
# 4. SEARCH FOR THE BEST 160-VIGNETTE DESIGN
# ============================================================

number_of_vignettes <- 160

number_of_design_searches <- 200

# Create a list in which the candidate designs will be stored.
candidate_designs <- vector(
  mode = "list",
  length = number_of_design_searches
)

# Create a data frame in which the D-efficiency of each
# candidate design will be stored.
design_diagnostics <- data.frame(
  search = seq_len(number_of_design_searches),
  D_efficiency = NA_real_
)

# Generate and evaluate the candidate designs.
for (i in seq_len(number_of_design_searches)) {

  # Use a different but reproducible seed for each search.
  set.seed(10000 + i)

  # Generate one D-efficient candidate design.
  candidate <- AlgDesign::optFederov(
    frml = design_formula,
    data = vignette_universe,
    nTrials = number_of_vignettes,
    approximate = FALSE,
    criterion = "D",
    maxIteration = 200,
    nRepeats = 1
  )

  # Identify the rows selected from the vignette universe.
  selected_rows <- candidate$rows

  # Retrieve the selected 160 vignettes.
  candidate_design <- vignette_universe[
    selected_rows,
    ,
    drop = FALSE
  ]

  # Store the complete candidate design.
  candidate_designs[[i]] <- candidate_design

  # Calculate and store its D-efficiency.
  design_diagnostics$D_efficiency[i] <-
    calculate_d_efficiency(
      candidate_design,
      design_formula
    )

  # Display the progress of the search.
  message(
    "Completed design search ",
    i,
    " of ",
    number_of_design_searches
  )
}

Completed design search 1 of 200

Completed design search 2 of 200

Completed design search 3 of 200

Completed design search 4 of 200

Completed design search 5 of 200

Completed design search 6 of 200

Completed design search 7 of 200

Completed design search 8 of 200

Completed design search 9 of 200

Completed design search 10 of 200

Completed design search 11 of 200

Completed design search 12 of 200

Completed design search 13 of 200

Completed design search 14 of 200

Completed design search 15 of 200

Completed design search 16 of 200

Completed design search 17 of 200

Completed design search 18 of 200

Completed design search 19 of 200

Completed design search 20 of 200

Completed design search 21 of 200

Completed design search 22 of 200

Completed design search 23 of 200

Completed design search 24 of 200

Completed design search 25 of 200

Completed design search 26 of 200

Completed design search 27 of 200

Completed design search 28 of 200

Completed design search 29 of

**Selecting the best candidate design**

After generating and evaluating the candidate designs, we select the design with the highest observed D-efficiency. Among the 200 alternatives considered, this is the design that provides the greatest amount of statistical information for estimating the model specified earlier.

The function which.max() identifies the search with the highest D-efficiency. The corresponding set of 160 vignettes is then retrieved from the list of candidate designs and stored as the final selected design.

The selected design is the best candidate found across the completed searches. It is not necessarily the theoretical optimum among all possible sets of 160 vignettes because testing every possible design is computationally infeasible.

In [ ]:
# ============================================================
# 5. SELECT THE BEST CANDIDATE DESIGN
# ============================================================

# Identify the candidate design with the highest D-efficiency.
best_search <- which.max(
  design_diagnostics$D_efficiency
)

# Retrieve the corresponding set of 160 vignettes.
selected_design <- candidate_designs[[best_search]]

# Display the search number and D-efficiency of the selected design.
cat(
  "\nSelected search:",
  best_search,
  "\n"
)

cat(
  "D-efficiency:",
  design_diagnostics$D_efficiency[best_search],
  "\n"
)



Selected search: 76 
D-efficiency: 0.5091054 


**Examining the selected design**

After selecting the design with the highest D-efficiency, we examine its general characteristics. The summary reports the number of vignettes, the number of unique and duplicated combinations, the number of model parameters, the rank of the design matrix, D-efficiency, and the maximum and average absolute correlations among predictors.

A full-rank design indicates that all parameters in the specified model can be estimated independently. Lower correlations indicate greater orthogonality. Some correlations may occur between contrast columns representing levels of the same categorical dimension and should not be interpreted as associations between different vignette dimensions.

In [ ]:
# ============================================================
# 5B. EXAMINE THE SELECTED DESIGN
# ============================================================

# Convert the selected design into a numerical model matrix
# using the same contrasts used to calculate D-efficiency.
selected_matrix <- model.matrix(
  design_formula,
  data = selected_design,
  contrasts.arg = lapply(
    selected_design[names(factors)],
    function(x) contr.sum(nlevels(x))
  )
)

# Remove the intercept before calculating correlations.
predictor_matrix <- selected_matrix[
  ,
  colnames(selected_matrix) != "(Intercept)",
  drop = FALSE
]

# Calculate the correlation matrix.
correlation_matrix <- cor(predictor_matrix)

# Extract the correlations between different columns.
correlation_values <- correlation_matrix[
  upper.tri(correlation_matrix)
]

# Produce a summary of the selected design.
design_summary <- data.frame(
  metric = c(
    "Number of vignettes",
    "Number of unique vignettes",
    "Number of duplicated vignettes",
    "Number of model parameters",
    "Rank of the design matrix",
    "Full rank",
    "D-efficiency",
    "Maximum absolute correlation",
    "Mean absolute correlation"
  ),

  value = c(
    nrow(selected_design),
    n_distinct(selected_design$vignette_universe_id),
    sum(duplicated(selected_design$vignette_universe_id)),
    ncol(selected_matrix),
    qr(selected_matrix)$rank,
    qr(selected_matrix)$rank == ncol(selected_matrix),
    design_diagnostics$D_efficiency[best_search],
    max(abs(correlation_values), na.rm = TRUE),
    mean(abs(correlation_values), na.rm = TRUE)
  )
)

# Print the general design metrics.
print(design_summary)

# Print the correlation matrix rounded to three decimal places.
print(
  round(correlation_matrix, 3)
)

                          metric        value
1            Number of vignettes 160.00000000
2     Number of unique vignettes 160.00000000
3 Number of duplicated vignettes   0.00000000
4     Number of model parameters  17.00000000
5      Rank of the design matrix  17.00000000
6                      Full rank   1.00000000
7                   D-efficiency   0.50910539
8   Maximum absolute correlation   0.50475447
9      Mean absolute correlation   0.05599789
                gender1   age1 age2   age3 education1 tenure1 children1
gender1               1  0.000  0.0  0.000      0.000       0       0.0
age1                  0  1.000  0.5  0.500      0.000       0       0.0
age2                  0  0.500  1.0  0.500      0.000       0       0.0
age3                  0  0.500  0.5  1.000      0.000       0       0.0
education1            0  0.000  0.0  0.000      1.000       0       0.0
tenure1               0  0.000  0.0  0.000      0.000       1       0.0
children1             0  0.000  0.0 

**Assigning the vignettes to decks**

The 160 vignettes selected through D-efficiency are randomly assigned to 10 decks of 16 vignettes each. The stopifnot() command verifies that the number and size of the decks correspond to the total number of selected vignettes.

The vignettes are first randomly reordered and then assigned a deck number, a position within the deck, and a unique vignette ID. The random seed makes this allocation reproducible.

In more advanced applications, deck allocation can be optimized to improve the balance of factor levels and reduce correlations within individual decks. For simplicity, this exercise uses random allocation.

In [ ]:
# ============================================================
# 6. ASSIGN THE 160 VIGNETTES TO 10 DECKS OF 16
# ============================================================

number_of_decks <- 10
vignettes_per_deck <- 16

# Check that the requested deck structure matches the total
# number of selected vignettes.
stopifnot(
  number_of_decks * vignettes_per_deck ==
    nrow(selected_design)
)

set.seed(20260825)

selected_design <- selected_design %>%
  slice_sample(prop = 1) %>%
  mutate(
    deck = rep(
      seq_len(number_of_decks),
      each = vignettes_per_deck
    )
  ) %>%
  group_by(deck) %>%
  mutate(
    vignette_in_deck = row_number()
  ) %>%
  ungroup() %>%
  mutate(
    vignette_id = row_number()
  )


**Creating the vignette text**

This section transforms the selected combinations of factor levels into readable vignette descriptions. First, each factor level is assigned a numeric code based on its position in the original list of levels. These variables will later facilitate data analysis.

The code then creates gender-consistent terms such as man/woman, he/she, and his/her. Conditional rules translate the remaining factor levels into natural-language phrases covering education, job tenure, performance, the firm’s economic situation, children, and earnings.

Finally, the four sentences are combined into one complete vignette. The dataset retains both the separate sentences and the full text.

In [ ]:
# ============================================================
# 7. CREATE THE VIGNETTE SENTENCES
# ============================================================

final_vignettes <- selected_design %>%
  mutate(

    # --------------------------------------------------------
    # Numeric value of each factor level
    # --------------------------------------------------------

    gender_n = as.integer(gender),
    age_n = as.integer(age),
    education_n = as.integer(education),
    tenure_n = as.integer(tenure),
    children_n = as.integer(children),
    performance_n = as.integer(performance),
    firm_situation_n = as.integer(firm_situation),
    earnings_n = as.integer(earnings),

    # --------------------------------------------------------
    # Gender-specific words
    # --------------------------------------------------------

    person_word = if_else(
      gender == "Male",
      "man",
      "woman"
    ),

pronoun = if_else(
  gender == "Male",
  "He",
  "She"
),

possessive = if_else(
  gender == "Male",
  "his",
  "her"
),

# --------------------------------------------------------
# Sentence 1: age, gender and education
# --------------------------------------------------------

sentence_1 = case_when(

  education == "Vocational training" ~
    paste0(
      "This is a ",
      age,
      "-year-old ",
      person_word,
      " who has completed vocational training."
    ),

  education == "University degree" ~
    paste0(
      "This is a ",
      age,
      "-year-old ",
      person_word,
      " who has a university degree."
    )
),

# --------------------------------------------------------
# Sentence 2: tenure, performance and firm
# --------------------------------------------------------

tenure_text = case_when(
  tenure == "Short" ~ "a short time",
  tenure == "Long" ~ "a long time"
),

performance_text = case_when(
  performance == "Below average" ~ "below average",
  performance == "Above average" ~ "above average"
),

firm_text = case_when(
  firm_situation == "High profits" ~
    "is highly profitable",

  firm_situation == "Solid" ~
    "is in a solid economic situation",

  firm_situation == "Threatened by bankruptcy" ~
    "is threatened by bankruptcy"
),

sentence_2 = paste0(
  pronoun,
  " has worked in ",
  possessive,
  " current job for ",
  tenure_text,
  ", performs ",
  performance_text,
  ", and works for a firm that ",
  firm_text,
  "."
),

# --------------------------------------------------------
# Sentence 3: children
# --------------------------------------------------------

sentence_3 = case_when(
  children == "0" ~
    paste0(pronoun, " has no children."),

  children == "1" ~
    paste0(pronoun, " has one child."),

  children == "2" ~
    paste0(pronoun, " has two children."),

  children == "3" ~
    paste0(pronoun, " has three children.")
),

# --------------------------------------------------------
# Sentence 4: earnings
# --------------------------------------------------------

sentence_4 = paste0(
  pronoun,
  " earns €",
  format(
    as.numeric(as.character(earnings)),
    big.mark = ",",
    scientific = FALSE,
    trim = TRUE
  ),
  " gross per month."
),

# --------------------------------------------------------
# Complete vignette
# --------------------------------------------------------

full_vignette = paste(
  sentence_1,
  sentence_2,
  sentence_3,
  sentence_4
)
)

**Checking the distribution of factor levels**

This section counts how frequently each factor level appears within each deck. The resulting table makes it possible to inspect whether gender, age, education, tenure, children, performance, firm situation, and earnings are reasonably distributed across the 10 decks.

Because the vignettes were randomly allocated to decks, the frequencies will not necessarily be identical. This table is therefore used as a diagnostic check and does not modify the design.

In [ ]:

# ============================================================
# 8. LEVEL DISTRIBUTIONS
# ============================================================

factor_names <- names(factors)

level_balance <- map_dfr(
  factor_names,
  function(variable) {

    final_vignettes %>%
      count(
        deck,
        level = .data[[variable]],
        name = "frequency"
      ) %>%
      mutate(
        dimension = variable,
        .before = 1
      )
  }
)

# Print the complete level-distribution table.
print(
  level_balance,
  n = Inf
)

# A tibble: 236 × 4
    dimension       deck level                    frequency
    <chr>          <int> <fct>                        <int>
  1 gender             1 Male                             7
  2 gender             1 Female                           9
  3 gender             2 Male                            10
  4 gender             2 Female                           6
  5 gender             3 Male                            12
  6 gender             3 Female                           4
  7 gender             4 Male                             9
  8 gender             4 Female                           7
  9 gender             5 Male                             3
 10 gender             5 Female                          13
 11 gender             6 Male                             4
 12 gender             6 Female                          12
 13 gender             7 Male                            11
 14 gender             7 Female                           5
 15 gender          

**Preparing and exporting the final design**

This section prepares the final dataset for export. Each row represents one vignette and includes its unique ID, deck number, position within the deck, numeric values for all eight dimensions, the four separate sentences, and the complete vignette text.

The output is saved as an Excel file containing one worksheet named Vignette text. Replace the path in output_file with the folder where you want to save the file. If a file with the same name already exists, overwrite = TRUE will replace it. Finally, R prints the file location to confirm that the export was completed.

In [ ]:
# ============================================================
# 9. PREPARE AND EXPORT THE EXCEL OUTPUTS
# ============================================================

# ------------------------------------------------------------
# FILE 1: ONE ROW PER VIGNETTE FOR DATA ANALYSIS
# ------------------------------------------------------------

analysis_output <- final_vignettes %>%
  select(
    vignette_id,
    deck,
    vignette_in_deck,

    # Numeric values of the dimensions
    gender_n,
    age_n,
    education_n,
    tenure_n,
    children_n,
    performance_n,
    firm_situation_n,
    earnings_n,

    # Vignette text
    sentence_1,
    sentence_2,
    sentence_3,
    sentence_4,
    full_vignette
  )


# ------------------------------------------------------------
# FILE 2: ONE ROW PER DECK FOR GENERATING WORD DOCUMENTS
# ------------------------------------------------------------

word_output <- final_vignettes %>%
  arrange(
    deck,
    vignette_in_deck
  ) %>%

  # Create positions 01, 02, ..., 16.
  mutate(
    vignette_position = sprintf(
      "%02d",
      vignette_in_deck
    )
  ) %>%

  # Keep the deck number and sentence columns.
  select(
    deck,
    vignette_position,
    sentence_1,
    sentence_2,
    sentence_3,
    sentence_4
  ) %>%

  # Transform the data from 16 rows per deck
  # to one row per deck.
  pivot_wider(
    names_from = vignette_position,
    values_from = c(
      sentence_1,
      sentence_2,
      sentence_3,
      sentence_4
    ),
    names_glue = "{.value}_{vignette_position}"
  ) %>%

  # Organize all sentence 1 columns first, followed by
  # sentence 2, sentence 3, and sentence 4.
  select(
    deck,
    starts_with("sentence_1_"),
    starts_with("sentence_2_"),
    starts_with("sentence_3_"),
    starts_with("sentence_4_")
  )


# ------------------------------------------------------------
# EXPORT BOTH EXCEL FILES IN GOOGLE COLAB
# ------------------------------------------------------------

analysis_file <- "/content/vignette_design_for_analysis.xlsx"

word_file <- "/content/vignette_decks_for_word.xlsx"

openxlsx::write.xlsx(
  analysis_output,
  file = analysis_file,
  sheetName = "Vignettes",
  overwrite = TRUE
)

openxlsx::write.xlsx(
  word_output,
  file = word_file,
  sheetName = "Decks",
  overwrite = TRUE
)
